# 06. Ensemble Evaluation

In [ ]:
# Common setup
from pathlib import Path
import sys
import os
import json
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Find project root: works when notebook is run from repo root or from notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    for parent in Path.cwd().resolve().parents:
        if (parent / "src").exists():
            PROJECT_ROOT = parent
            break

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = PROJECT_ROOT / "data"
REPORT_DIR = PROJECT_ROOT / "reports"
FIG_DIR = REPORT_DIR / "figures"
TABLE_DIR = REPORT_DIR / "tables"
EXP_DIR = PROJECT_ROOT / "experiments" / "notebook_outputs"
for p in [FIG_DIR, TABLE_DIR, EXP_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Source path:", SRC_DIR)

import torch
from torch.utils.data import DataLoader

from bioacoustic.dataset import read_metadata, build_class_list, add_stratified_folds, BirdAudioDataset, encode_multihot, make_label_map
from bioacoustic.ensemble import average_predictions, blend_regular_shifted, temporal_smoothing, power_adjust
from bioacoustic.metrics import compute_multilabel_metrics, search_best_threshold, per_class_metrics
from bioacoustic.openvino_ensemble import OpenVINOEnsemble, find_openvino_models, infer_audio_file_openvino
from bioacoustic.audio import load_audio, segment_for_inference
from bioacoustic.spectrogram import batch_log_mel
from bioacoustic.utils import seed_everything

seed_everything(42)

## Configuration

In [ ]:
CFG = {
    "metadata_path": DATA_DIR / "train_metadata.csv",
    "audio_dir": DATA_DIR / "train_audio",
    "output_dir": EXP_DIR / "ensemble",
    "prediction_dirs": [
        EXP_DIR / "baseline",
        EXP_DIR / "sed",
        EXP_DIR / "self_distillation",
    ],
    "openvino_models_dir": PROJECT_ROOT / "weights" / "openvino",
    "sample_rate": 32000,
    "duration": 5.0,
    "n_fft": 2048,
    "hop_length": 768,
    "n_mels": 192,
    "f_min": 50,
    "f_max": 15000,
    "fold": 0,
    "n_splits": 5,
    "gamma": 1.0,
    "max_valid_files_for_openvino": 200,
}
CFG["output_dir"].mkdir(parents=True, exist_ok=True)

## Mode A: ensemble existing prediction files

In [ ]:
pred_list = []
y_true = None
source_names = []
for d in CFG["prediction_dirs"]:
    pred_path = Path(d) / "valid_y_pred.npy"
    true_path = Path(d) / "valid_y_true.npy"
    if pred_path.exists() and true_path.exists():
        pred = np.load(pred_path)
        true = np.load(true_path)
        pred_list.append(pred)
        source_names.append(Path(d).name)
        if y_true is None:
            y_true = true
        else:
            assert y_true.shape == true.shape, f"Target shape mismatch for {d}"

print("Loaded prediction sources:", source_names)
if pred_list:
    min_n = min(p.shape[0] for p in pred_list)
    pred_list = [p[:min_n] for p in pred_list]
    y_true = y_true[:min_n]
    ensemble_pred = average_predictions(pred_list)
    ensemble_pred = power_adjust(ensemble_pred, gamma=CFG["gamma"])
    metrics_05 = compute_multilabel_metrics(y_true, ensemble_pred, threshold=0.5)
    best_t, best_f1 = search_best_threshold(y_true, ensemble_pred)
    metrics_best = compute_multilabel_metrics(y_true, ensemble_pred, threshold=best_t)

    print("Metrics @ 0.5 threshold")
    display(pd.DataFrame([metrics_05]).T.rename(columns={0: "value"}))
    print("Best threshold:", best_t, "Best macro F1:", best_f1)
    display(pd.DataFrame([metrics_best]).T.rename(columns={0: "value"}))

    pd.DataFrame([metrics_05]).to_csv(CFG["output_dir"] / "ensemble_metrics_threshold_05.csv", index=False)
    pd.DataFrame([metrics_best]).to_csv(CFG["output_dir"] / "ensemble_metrics_best_threshold.csv", index=False)
    np.save(CFG["output_dir"] / "ensemble_y_pred.npy", ensemble_pred)
    np.save(CFG["output_dir"] / "ensemble_y_true.npy", y_true)
else:
    print("No existing prediction files found. Run model experiment notebooks first or use OpenVINO mode below.")

## Compare individual models vs ensemble

In [ ]:
if pred_list:
    rows = []
    for name, pred in zip(source_names, pred_list):
        m = compute_multilabel_metrics(y_true, pred, threshold=0.5)
        m["setting"] = name
        rows.append(m)
    m = compute_multilabel_metrics(y_true, ensemble_pred, threshold=0.5)
    m["setting"] = "ensemble"
    rows.append(m)
    compare_df = pd.DataFrame(rows)
    display(compare_df[["setting", "macro_auc", "macro_map", "precision_macro", "recall_macro", "f1_macro"]])
    compare_df.to_csv(CFG["output_dir"] / "individual_vs_ensemble_metrics.csv", index=False)

    plot_df = compare_df.set_index("setting")[["macro_auc", "macro_map", "f1_macro"]]
    plot_df.plot(kind="bar", figsize=(9, 4))
    plt.title("Individual models vs ensemble")
    plt.ylabel("Metric")
    plt.ylim(0, 1)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "individual_vs_ensemble_metrics.png", dpi=200)
    plt.show()

## Per-class analysis

In [ ]:
if pred_list:
    df_meta = read_metadata(CFG["metadata_path"])
    classes = build_class_list(df_meta)
    pc = pd.DataFrame(per_class_metrics(y_true, ensemble_pred, classes))
    display(pc.head())
    pc.to_csv(CFG["output_dir"] / "per_class_metrics.csv", index=False)

    # Plot classes with highest support and lowest AP among valid classes.
    valid_pc = pc.dropna(subset=["ap"]).copy()
    if len(valid_pc):
        worst = valid_pc.sort_values("ap").head(20)
        plt.figure(figsize=(10, 5))
        plt.bar(worst["class"].astype(str), worst["ap"])
        plt.title("Lowest per-class average precision")
        plt.ylabel("Average precision")
        plt.xticks(rotation=60, ha="right")
        plt.tight_layout()
        plt.savefig(FIG_DIR / "lowest_per_class_ap.png", dpi=200)
        plt.show()

## Mode B: OpenVINO ensemble on validation audio

Use this when `weights/openvino/` contains `model_*.xml` and optionally `model2_*.xml`. This is slower but evaluates the actual OpenVINO inference pipeline.

In [ ]:
ov_dir = Path(CFG["openvino_models_dir"])
regular_paths = find_openvino_models(ov_dir, prefix="model_") if ov_dir.exists() else []
shifted_paths = find_openvino_models(ov_dir, prefix="model2_") if ov_dir.exists() else []
print("Regular OpenVINO models:", len(regular_paths))
print("Shifted OpenVINO models:", len(shifted_paths))

if regular_paths:
    df = read_metadata(CFG["metadata_path"])
    classes = build_class_list(df)
    label_map = make_label_map(classes)
    if "fold" not in df.columns:
        df = add_stratified_folds(df, n_splits=CFG["n_splits"], seed=42)
    valid_df = df[df["fold"] == CFG["fold"]].reset_index(drop=True)
    if CFG["max_valid_files_for_openvino"]:
        valid_df = valid_df.sample(min(CFG["max_valid_files_for_openvino"], len(valid_df)), random_state=42).reset_index(drop=True)

    regular_ens = OpenVINOEnsemble(regular_paths)
    shifted_ens = OpenVINOEnsemble(shifted_paths) if shifted_paths else None
    spec_kwargs = dict(n_fft=CFG["n_fft"], hop_length=CFG["hop_length"], n_mels=CFG["n_mels"], f_min=CFG["f_min"], f_max=CFG["f_max"])

    preds, targets, row_ids = [], [], []
    for _, row in valid_df.iterrows():
        filename = str(row["filename"])
        p1 = Path(CFG["audio_dir"]) / filename
        p2 = Path(CFG["audio_dir"]) / str(row["primary_label"]) / filename
        audio_path = p1 if p1.exists() else p2
        if not audio_path.exists():
            continue
        chunk_pred = infer_audio_file_openvino(
            audio_path,
            regular_ens,
            shifted_ens,
            sample_rate=CFG["sample_rate"],
            duration=CFG["duration"],
            shift=2.5,
            blend_alpha=0.5,
            smooth=True,
            spectrogram_kwargs=spec_kwargs,
        )
        # Training labels are clip-level weak labels. For validation approximation,
        # assign the same multi-hot target to each generated chunk.
        target = encode_multihot(row["primary_label"], row.get("secondary_labels", None), label_map, include_secondary=True)
        preds.append(chunk_pred)
        targets.append(np.repeat(target[None, :], len(chunk_pred), axis=0))
        for i in range(len(chunk_pred)):
            row_ids.append(f"{Path(filename).stem}_{(i+1)*int(CFG['duration'])}")

    if preds:
        ov_y_pred = np.concatenate(preds, axis=0)
        ov_y_true = np.concatenate(targets, axis=0)
        ov_metrics = compute_multilabel_metrics(ov_y_true, ov_y_pred, threshold=0.5)
        display(pd.DataFrame([ov_metrics]).T.rename(columns={0: "value"}))
        np.save(CFG["output_dir"] / "openvino_y_pred.npy", ov_y_pred)
        np.save(CFG["output_dir"] / "openvino_y_true.npy", ov_y_true)
        pd.DataFrame([ov_metrics]).to_csv(CFG["output_dir"] / "openvino_ensemble_metrics.csv", index=False)
    else:
        print("No validation audio files could be evaluated.")
else:
    print("OpenVINO model files not found. Skipping OpenVINO mode.")